In [38]:
# Local steps to run in pycharm
# conda activate sage
# jupyter lab --no-browser --port=8888
# configure external server with the token
# sagemath (10.7)

from sage.all import *
from sage.plot.plot import graphics_array
from itertools import combinations
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def graph_square(H):
    G = Graph()
    G.add_vertices(H.vertices())
    for u in H.vertices():
        for v in H.vertices():
            if u < v and H.distance(u,v) <= 2:
                G.add_edge(u,v)
    return G

def is_square_root(G, H):
    return graph_square(H) == G

def find_square_root(G):
    V = list(G.vertices())
    n = len(V)
    pairs = list(combinations(range(n), 2))

    total = 1 << len(pairs)
    for mask in range(total):
        H = Graph()
        H.add_vertices(V)
        for bit, (i, j) in enumerate(pairs):
            if (mask >> bit) & 1:
                H.add_edge(V[i], V[j])
        if is_square_root(G, H):
            return H
    return None

In [39]:
# Efficient find_square_root function

def violates_partial(G, H):
    for u, v in combinations(H.vertices(), 2):
        try:
            d = H.distance(u, v)
        except (ValueError, RuntimeError):
            continue

        if H.has_edge(u, v) and not G.has_edge(u, v):
            return True
        if d <= 2 and not G.has_edge(u, v):
            return True
    return False

def find_square_root(G):
    V = list(G.vertices())
    pairs = list(combinations(V, 2))
    H = Graph()
    H.add_vertices(V)

    def backtrack(idx):
        if idx == len(pairs):
            return H.copy() if is_square_root(G, H) else None

        u, v = pairs[idx]

        H.add_edge(u, v)
        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result
        H.delete_edge(u, v)

        if not violates_partial(G, H):
            result = backtrack(idx + 1)
            if result:
                return result

        return None

    return backtrack(0)

In [40]:
# Generate linear program
def linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(integer=True, nonnegative=True)

    A_G = G.adjacency_matrix()

    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j])
            else:
                m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) + x[i,j])

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] <= M * A_G[i][j])
            m.add_constraint(z[i,j] >= A_G[i][j])

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] <= x[i,k])
                m.add_constraint(y[i,k,j] <= x[k,j])
                m.add_constraint(y[i,k,j] >= x[i,k] + x[k,j] - 1)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] <= 1)
            for k in range(n):
                m.add_constraint(y[i,k,j] <= 1)

    m.set_objective(0)
    return m, x

# LP with max objective function
def max_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=True)
    m.solver_parameter("simplex_or_intopt", "intopt_only")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j])
            else:
                m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) + x[i,j])

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] <= M * A_G[i][j])
            m.add_constraint(z[i,j] >= A_G[i][j])

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] <= x[i,k])
                m.add_constraint(y[i,k,j] <= x[k,j])
                m.add_constraint(y[i,k,j] >= x[i,k] + x[k,j] - 1)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] <= 1)
            for k in range(n):
                m.add_constraint(y[i,k,j] <= 1)

    m.set_objective(m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

# LP with min objective function with gomory cuts
def min_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")
    m.solver_parameter("gomory_cuts", "GLP_ON")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

    # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    m.set_objective(- m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

In [41]:
# LP with max objective function and additional triangle constraints for line graph root structure
def max_triangular_linear_program(G):
    m = MixedIntegerLinearProgram(solver = 'GLPK', maximization=False)
    m.solver_parameter("simplex_or_intopt", "intopt_only")
    # m.solver_parameter("gomory_cuts", "GLP_ON")

    x = m.new_variable(real=True, nonnegative=True)
    y = m.new_variable(real=True, nonnegative=True)
    z = m.new_variable(real=True, nonnegative=True)

    A_G = G.adjacency_matrix()
    n = A_G.nrows()
    M = n**2

        # Indicator argument
    for i in range(n):
        for j in range(n):
            if i == j:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j]) == 0)
            else:
                m.add_constraint(z[i,j] - (sum(y[i,k,j] for k in range(n)) + x[i,j]) == 0)

    # Indicator function
    for i in range(n):
        for j in range(n):
            m.add_constraint(z[i,j] - M * A_G[i][j] <= 0)
            m.add_constraint(-z[i,j] + A_G[i][j] <= 0)

    # Product
    for i in range(n):
        for j in range(n):
            for k in range(n):
                m.add_constraint(y[i,k,j] - x[i,k] <= 0)
                m.add_constraint(y[i,k,j] - x[k,j] <= 0)
                m.add_constraint(- y[i,k,j] + x[i,k] + x[k,j] - 1 <= 0)

    # Symmetry
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] == x[j,i])

    # Diagonal
    for i in range(n):
        m.add_constraint(x[i,i] == 0)

    # Variables
    for i in range(n):
        for j in range(n):
            m.add_constraint(x[i,j] - 1 <= 0)
            for k in range(n):
                m.add_constraint(y[i,k,j] - 1 <= 0)

    # Count triangles and consider vertices in only one triangle O(n^3)
    deg = {v: G.degree(v) for v in range(n)}
    triangles_at = {v: [] for v in range(n)}

    for i in range(n):
        for j in range(i+1, n):
            if not G.has_edge(i, j):
                continue
            for k in range(j+1, n):
                if G.has_edge(i, k) and G.has_edge(j, k):
                    triangles_at[i].append((j, k))
                    triangles_at[j].append((i, k))
                    triangles_at[k].append((i, j))

    for s, tris in triangles_at.items():
        if len(tris) == 1:
            u, v = tris[0]
            if deg[u] < deg[v]:
                w = u
            elif deg[v] < deg[u]:
                w = v
            else:
                continue
            m.add_constraint(x[s,w] == 1)
            m.add_constraint(x[w,s] == 1)

    m.set_objective(-m.sum(x[i,j] for i in range(n) for j in range(n)))
    return m, x

# LP with max objective function and additional triangle constraints for line graph root structure.
# If triangle is undetermined due to equality, only add one such assignment recursively.
def max_recursive_triangular_linear_program(G, max_iters=None):
    n = G.order()
    if max_iters is None:
        max_iters = n * n

    deg = {v: G.degree(v) for v in range(n)}
    triangles_at = {v: [] for v in range(n)}

    for i in range(n):
        for j in range(i + 1, n):
            if not G.has_edge(i, j):
                continue
            for k in range(j + 1, n):
                if G.has_edge(i, k) and G.has_edge(j, k):
                    triangles_at[i].append((j, k))
                    triangles_at[j].append((i, k))
                    triangles_at[k].append((i, j))

    used_ties = set()

    for _ in range(max_iters):

        m = MixedIntegerLinearProgram(maximization=True)
        x = m.new_variable(real=True)
        y = m.new_variable(real=True)
        z = m.new_variable(real=True)

        A_G = G.adjacency_matrix()
        M = n**2

        # Variables
        for i in range(n):
            for j in range(n):
                m.add_constraint(x[i,j] >= 0)
                m.add_constraint(x[i,j] <= 1)
                m.add_constraint(z[i,j] >= 0)
                for k in range(0, n):
                    m.add_constraint(y[i,k,j] >= 0)
                    m.add_constraint(y[i,k,j] <= 1)

        # Indicator argument
        for i in range(n):
            for j in range(n):
                if i == j:
                    m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) - sum(x[i,k] for k in range(n)) + x[i,j])
                else:
                    m.add_constraint(z[i,j] == sum(y[i,k,j] for k in range(n)) + x[i,j])

        # Indicator function
        for i in range(n):
            for j in range(n):
                m.add_constraint(z[i,j] <= M * A_G[i][j])
                m.add_constraint(z[i,j] >= A_G[i][j])

        # Product
        for i in range(n):
            for j in range(n):
                for k in range(n):
                    m.add_constraint(y[i,k,j] <= x[i,k])
                    m.add_constraint(y[i,k,j] <= x[k,j])
                    m.add_constraint(y[i,k,j] >= x[i,k] + x[k,j] - 1)

        # Symmetry
        for i in range(n):
            for j in range(n):
                m.add_constraint(x[i,j] == x[j,i])
                m.add_constraint(z[i,j] == z[j,i])

        # Diagonal
        for i in range(n):
            m.add_constraint(x[i,i] == 0)
            m.add_constraint(z[i,i] == 0)

        new_constraint_added = False
        tied_candidates = []

        for s, tris in triangles_at.items():
            if len(tris) != 1:
                continue

            u, v = tris[0]

            if deg[u] < deg[v]:
                w = u
                m.add_constraint(x[s,w] == 1)
                m.add_constraint(x[w,s] == 1)
                new_constraint_added = True

            elif deg[v] < deg[u]:
                w = v
                m.add_constraint(x[s,w] == 1)
                m.add_constraint(x[w,s] == 1)
                new_constraint_added = True

            else:
                key = (s, min(u, v), max(u, v))
                if key not in used_ties:
                    tied_candidates.append((s, u, v))

        if tied_candidates:
            tied_candidates.sort()
            s, u, v = tied_candidates[0]
            used_ties.add((s, min(u, v), max(u, v)))

            w = min(u, v)
            m.add_constraint(x[s,w] == 1)
            m.add_constraint(x[w,s] == 1)
            new_constraint_added = True

        m.set_objective(m.sum(x[i,j] for i in range(n) for j in range(n)))

        try:
            m.solve()
        except:
            return m, x

        if not new_constraint_added:
            return m, x

    # Iteration cap reached (still safe)
    return m, x

In [51]:
# Generate all possible graph squares/roots for given n
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sage.numerical.mip import MIPSolverException
from sage.graphs.graph_generators import graphs as graph_gens

n = 7
ncpus = 12
dir = f"max_triangular_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment='center')
    g.axes(False)
    return g

def process_graph(idx, G):
    G.relabel(range(G.order()))
    n = G.order()

    A_H = None
    sol_H = None
    H = None
    lp_status = "unknown"
    has_root = False
    edges_H = None
    is_noninteger = False

    try:
        m, x = max_triangular_linear_program(G)
        m.solve()
        solution_x = m.get_values(x)
        A_H = matrix([[solution_x[i, j] for j in range(n)] for i in range(n)])
        sol_H = Graph(A_H, weighted=True)
        for (u, v, w) in sol_H.edges():
            sol_H.set_edge_label(u, v, round(w, 3))
        p1 = sol_H.plot(edge_labels=True, title='Weighted LP H')
        lp_status = "feasible"

        if any(abs(A_H[i, j] - round(A_H[i, j])) > 1e-6 for i in range(n) for j in range(n)):
            is_noninteger = True

    except MIPSolverException:
        sol_H = None
        p1 = error_plot("LP infeasible")
        lp_status = "infeasible"

    H = find_square_root(G)

    if H is None:
        p2 = error_plot("No feasible H")
        has_root = False
        edges_H = None
    else:
        p2 = H.plot(title='Feasible H')
        has_root = True
        edges_H = H.size()


    p3 = G.plot(title='Original G')
    combined = graphics_array([p1, p2, p3])

    summary = {
        "index": idx,
        "edges_G": G.size(),
        "vertices_G": n,
        "lp_status": lp_status,
        "has_root": has_root,
        "edges_H": edges_H,
        "is_noninteger": is_noninteger
    }

    is_mismatch = (is_noninteger or (lp_status == "feasible" and H is None))

    if is_mismatch:
        if sol_H is not None:
            p_left = sol_H.plot(edge_labels=True, title="Weighted LP H")
        else:
            p_left = error_plot("No LP solution")

        if H is not None:
            p_mid = H.plot(title="Feasible H")
        else:
            p_mid = error_plot("No discrete H")

        p_right = G.plot(title="Original G")
        comparison = graphics_array([p_left, p_mid, p_right])
    else:
        comparison = None
    return combined, summary, comparison, is_mismatch

def run_task(idx, G):
    return process_graph(idx, G)

tasks = [(i, Graph(G0)) for i, G0 in enumerate(graph_gens.nauty_geng(str(n)))]

results = []
mismatch_results = []

for idx, G in tasks:
    plot, summary, comparison, is_mismatch = run_task(idx, G)
    results.append((plot, summary))
    if is_mismatch and comparison is not None:
        mismatch_results.append((comparison, summary))
    print(f"✅ Graph {summary['index']:>4}: LP={summary['lp_status']} Root={summary['has_root']} "
          f"{'(⚠️ mismatch)' if is_mismatch else ''}")

with open(summary_file, "w") as f:
    header = f"{'Graph':>5} | {'Verts':>5} | {'Edges(G)':>8} | {'Edges(H)':>8} | {'LP':>10} | {'Root':>5} | {'NonInt':>7}\n"
    f.write(header)
    f.write("-" * len(header) + "\n")
    for _, s in results:
        f.write(
            f"{s['index']:>5} | "
            f"{s['vertices_G']:>5} | "
            f"{s['edges_G']:>8} | "
            f"{str(s['edges_H']):>8} | "
            f"{s['lp_status']:<10} | "
            f"{'Yes' if s['has_root'] else 'No':>5} | "
            f"{'Yes' if s['is_noninteger'] else 'No':>7}\n"
        )

print(f"📝 Summary written to {summary_file}")

with PdfPages(output_pdf) as pdf:
    for i, (p, s) in enumerate(results):
        fig = p.matplotlib(figsize=(10, 10))
        pdf.savefig(fig)
        plt.close(fig)
print(f"\n✅ Combined {len(results)} graphs into {output_pdf}")


if mismatch_results:
    with PdfPages(comparison_pdf) as pdf:
        for comp, s in mismatch_results:

            fig = comp.matplotlib(figsize=(12, 4))

            plt.suptitle(
                f"Graph {s['index']} — "
                f"LP={s['lp_status']} | "
                f"Root={s['has_root']} | "
                f"{'Non-integer LP' if s['is_noninteger'] else ''}"
            )

            pdf.savefig(fig)
            plt.close(fig)

    print(f"⚠️ Comparison mismatches saved to {comparison_pdf}")

✅ Graph    0: LP=feasible Root=True 
✅ Graph    1: LP=feasible Root=True 
✅ Graph    2: LP=infeasible Root=False 
✅ Graph    3: LP=infeasible Root=False 
✅ Graph    4: LP=infeasible Root=False 
✅ Graph    5: LP=infeasible Root=False 
✅ Graph    6: LP=infeasible Root=False 
✅ Graph    7: LP=feasible Root=True 
✅ Graph    8: LP=infeasible Root=False 
✅ Graph    9: LP=infeasible Root=False 
✅ Graph   10: LP=feasible Root=True 
✅ Graph   11: LP=infeasible Root=False 
✅ Graph   12: LP=infeasible Root=False 
✅ Graph   13: LP=infeasible Root=False 
✅ Graph   14: LP=infeasible Root=False 
✅ Graph   15: LP=infeasible Root=False 
✅ Graph   16: LP=infeasible Root=False 
✅ Graph   17: LP=infeasible Root=False 
✅ Graph   18: LP=infeasible Root=False 
✅ Graph   19: LP=infeasible Root=False 
✅ Graph   20: LP=infeasible Root=False 
✅ Graph   21: LP=infeasible Root=False 
✅ Graph   22: LP=infeasible Root=False 
✅ Graph   23: LP=infeasible Root=False 
✅ Graph   24: LP=infeasible Root=False 
✅ Graph   25

<Figure size 640x480 with 0 Axes>

In [52]:
# Analyze graph summary
filename = "max_triangular_graph_output/n=7/graph_summary_n=7.txt"

summary = {}

count_feasible_root_no = 0
violations = []

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith('-') or line.startswith('Graph'):
            continue

        parts = [p.strip() for p in line.split('|')]

        if len(parts) != 7:
            continue

        graph, verts, edges_g, edges_h, lp, root, nonint = parts

        if lp == "feasible" and root == "No":
            count_feasible_root_no += 1

        if lp == "infeasible" and root != "No":
            violations.append(parts)

        H_exists = edges_h.lower() != "none"

        if lp == "infeasible":
            LP_type = "LP infeasible"
        elif nonint == "Yes":
            LP_type = "LP fractional solution"
        else:
            LP_type = "LP integer solution"

        key = (H_exists, LP_type)
        summary.setdefault(key, []).append(graph)

print(f"\nFeasible & Root=No count: {count_feasible_root_no}")

if violations:
    print("\n⚠️ Violations found (LP infeasible but Root ≠ No):")
    for v in violations:
        print(v)
else:
    print("\n✅ All infeasible rows have Root=No (rule holds).")

print("\n===== Summary by (H_exists, LP_type) =====")
for (H_exists, LP_type), graphs in summary.items():
    print(
        f"H exists: {H_exists:<5} | "
        f"{LP_type:<25} → Count: {len(graphs):<3} | "
        f"Graphs: {', '.join(graphs)}"
    )


Feasible & Root=No count: 136

✅ All infeasible rows have Root=No (rule holds).

===== Summary by (H_exists, LP_type) =====
H exists: 1     | LP integer solution       → Count: 51  | Graphs: 0, 1, 7, 10, 25, 57, 61, 86, 101, 119, 159, 262, 276, 347, 369, 375, 380, 412, 417, 425, 521, 528, 615, 640, 648, 655, 666, 712, 766, 779, 830, 831, 871, 877, 916, 934, 937, 938, 940, 946, 948, 952, 980, 982, 1006, 1022, 1023, 1026, 1028, 1031, 1043
H exists: 0     | LP infeasible             → Count: 831 | Graphs: 2, 3, 4, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 58, 59, 60, 62, 63, 64, 65, 66, 67, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 1

In [44]:
# Build graph of integer edges and find connected components
# Rank vertices by how fractional their neighborhood is
# For the top vertex, rank its incident fractional edges
def integer_components_and_ranked_vertex_edges(x_val, n, eps=1e-6):
    G_int = Graph()
    G_int.add_vertices(range(n))

    for (i, j), v in x_val.items():
        if i < j and v > 1 - eps:
            G_int.add_edge(i, j)

    components = G_int.connected_components()

    comp_index = {}
    for idx, C in enumerate(components):
        for v in C:
            comp_index[v] = idx

    vertex_score = {}
    for (i, j), v in x_val.items():
        if i >= j:
            continue
        if eps < v < 1 - eps:
            w = abs(v - 0.5)
            vertex_score[i] = vertex_score.get(i, 0) + w
            vertex_score[j] = vertex_score.get(j, 0) + w

    if not vertex_score:
        return components, comp_index, None, []

    v_star = max(vertex_score.items(), key=lambda t: (t[1], -t[0]))[0]

    incident_edges = []
    for (i, j), v in x_val.items():
        if eps < v < 1 - eps:
            if i == v_star or j == v_star:
                a, b = min(i, j), max(i, j)
                incident_edges.append((a, b, v, abs(v - 0.5)))

    incident_edges.sort(key=lambda t: (-t[3], t[0], t[1]))

    return components, comp_index, v_star, incident_edges

# Rebuild linear program wit fixed assignments
def solve_lp_with_fixes(G, fixed_assignments):
    m, x = max_linear_program(G)
    for (i, j, val) in fixed_assignments:
        m.add_constraint(x[i, j] == val)
        m.add_constraint(x[j, i] == val)
    m.solve()
    return m, x

# Deterministic iterative rounding with bounded backtracking determined by the beam width (beam search) O(max_iters * beam_width * LP)
def max_rounding_linear_program(G, max_iters=100, threshold=0.5, beam_width=2):
    rounding_steps = []
    fixed_assignments = []
    ever_fractional = False
    n = G.order()

    try:
        m, x = max_linear_program(G)
        m.solve()
    except:
        return None, None, [], None, "lp_infeasible"

    x_val = m.get_values(x)
    A0 = matrix([[x_val[a,b] for b in range(n)] for a in range(n)])
    H0 = Graph(A0, weighted=True)
    for (u,v,w) in H0.edges():
        H0.set_edge_label(u,v,round(w,3))

    rounding_steps.append({
        "step": 0,
        "fixed": None,
        "value": None,
        "graph": H0,
        "label": "Initial LP solution"
    })

    for step in range(1, max_iters + 1):
        x_val = m.get_values(x)
        comps, comp_index, v_star, edges = integer_components_and_ranked_vertex_edges(x_val, n)

        if v_star is None:
            A = matrix(ZZ, [
                [int(round(x_val[a,b])) for b in range(n)]
                for a in range(n)
            ])
            H = Graph(A, weighted=False)
            state = "lp_fractional_rounded" if ever_fractional else "lp_integral"
            return m, x, rounding_steps, H, state

        ever_fractional = True
        success = False

        for (i, j, v, _) in edges[:beam_width]:
            ci, cj = comp_index[i], comp_index[j]

            if ci != cj:
                fix_val = 1
            else:
                fix_val = 1 if v > threshold else 0

            try:
                new_fixes = fixed_assignments + [(i,j,fix_val)]
                m_try, x_try = solve_lp_with_fixes(G, new_fixes)
                x_try_val = m_try.get_values(x_try)
            except:
                continue

            fixed_assignments = new_fixes
            m, x = m_try, x_try
            x_val = x_try_val

            A = matrix([[x_val[a,b] for b in range(n)] for a in range(n)])
            H = Graph(A, weighted=True)
            for (u,v2,w) in H.edges():
                H.set_edge_label(u,v2,round(w,3))

            rounding_steps.append({
                "step": step,
                "fixed": (i,j),
                "value": fix_val,
                "graph": H,
                "label": f"Rounding {step}: x[{i},{j}]={fix_val}"
            })

            success = True
            break

        if not success:
            return m, x, rounding_steps, None, "lp_fractional_infeasible"

    return m, x, rounding_steps, None, "lp_fractional_infeasible"

In [45]:
# Rebuild triangular linear program wit fixed assignments
def solve_triangular_lp_with_fixes(G, fixed_assignments):
    m, x = max_triangular_linear_program(G)
    for (i, j, val) in fixed_assignments:
        m.add_constraint(x[i, j] == val)
        m.add_constraint(x[j, i] == val)
    m.solve()
    return m, x

# Deterministic iterative rounding with bounded backtracking determined by the beam width (beam search) O(max_iters * beam_width * LP)
def max_rounding_triangular_linear_program(G, max_iters=100, threshold=0.5, beam_width=2):
    rounding_steps = []
    fixed_assignments = []
    ever_fractional = False
    n = G.order()

    try:
        m, x = max_triangular_linear_program(G)
        m.solve()
    except:
        return None, None, [], None, "lp_infeasible"

    x_val = m.get_values(x)
    A0 = matrix([[x_val[a,b] for b in range(n)] for a in range(n)])
    H0 = Graph(A0, weighted=True)
    for (u,v,w) in H0.edges():
        H0.set_edge_label(u,v,round(w,3))

    rounding_steps.append({
        "step": 0,
        "fixed": None,
        "value": None,
        "graph": H0,
        "label": "Initial LP solution"
    })

    for step in range(1, max_iters + 1):
        x_val = m.get_values(x)
        comps, comp_index, v_star, edges = integer_components_and_ranked_vertex_edges(x_val, n)

        if v_star is None:
            A = matrix(ZZ, [
                [int(round(x_val[a,b])) for b in range(n)]
                for a in range(n)
            ])
            H = Graph(A, weighted=False)
            state = "lp_fractional_rounded" if ever_fractional else "lp_integral"
            return m, x, rounding_steps, H, state

        ever_fractional = True
        success = False

        for (i, j, v, _) in edges[:beam_width]:
            ci, cj = comp_index[i], comp_index[j]

            if ci != cj:
                fix_val = 1
            else:
                fix_val = 1 if v > threshold else 0

            try:
                new_fixes = fixed_assignments + [(i,j,fix_val)]
                m_try, x_try = solve_triangular_lp_with_fixes(G, new_fixes)
                x_try_val = m_try.get_values(x_try)
            except:
                continue

            fixed_assignments = new_fixes
            m, x = m_try, x_try
            x_val = x_try_val

            A = matrix([[x_val[a,b] for b in range(n)] for a in range(n)])
            H = Graph(A, weighted=True)
            for (u,v2,w) in H.edges():
                H.set_edge_label(u,v2,round(w,3))

            rounding_steps.append({
                "step": step,
                "fixed": (i,j),
                "value": fix_val,
                "graph": H,
                "label": f"Rounding {step}: x[{i},{j}]={fix_val}"
            })

            success = True
            break

        if not success:
            return m, x, rounding_steps, None, "lp_fractional_infeasible"

    return m, x, rounding_steps, None, "lp_fractional_infeasible"

In [46]:
# Generate all possible graph squares and their tree roots for given n
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

n = 6
ncpus = 12
dir = f"max_rounding_triangular_graph_output"
output_dir = os.path.join(dir, f"n={n}")
output_pdf = os.path.join(output_dir, f"graph_results_n={n}.pdf")
comparison_pdf = os.path.join(output_dir, f"graph_comparison_n={n}.pdf")
summary_file = os.path.join(output_dir, f"graph_summary_n={n}.txt")

os.makedirs(output_dir, exist_ok=True)
os.makedirs(dir, exist_ok=True)

def error_plot(message):
    g = Graphics()
    g += text(message, (0, 0), fontsize=14, color="red", horizontal_alignment='center')
    g.axes(False)
    return g

def plot_with_labels(G, title):
    return G.plot(edge_labels=True, fontsize=8, title=title)

def classify_graph(summary):
    lp = summary["lp_status"]
    has_root = summary["has_root"]

    if lp == "integral":
        return "OK: H exists, LP integral" if has_root \
               else "ERROR: LP integral but no H"

    if lp == "fractional_rounded":
        return "OK: H exists via rounding" if has_root \
               else "ERROR: rounding found H but none exists"

    if lp == "fractional_infeasible":
        return "ERROR: H exists but rounding failed" if has_root \
               else "OK: rounding failed but no H"

    if lp == "infeasible":
        return "ERROR: H exists but LP infeasible" if has_root \
               else "OK: no H, LP infeasible"

    return "UNKNOWN"

def process_graph(idx, G):
    G.relabel(range(G.order()))
    n = G.order()

    sol_H = None
    H = None
    lp_status = "unknown"
    has_root = False
    edges_H = None
    rounding_steps = []

    # ---------- LP + rounding ----------
    try:
        m, x, rounding_steps, integral_H, lp_state = max_rounding_triangular_linear_program(G)
        lp_status = lp_state.replace("lp_", "")

        if lp_state == "lp_integral":
            sol_H = integral_H
            p1 = plot_with_labels(sol_H, "LP Integral H")

        elif lp_state == "lp_fractional_rounded":
            H_snap = rounding_steps[-1]["graph"]
            sol_H = H_snap
            p1 = plot_with_labels(sol_H, "LP Fractional → Rounded H")

        elif lp_state == "lp_fractional_infeasible":
            p1 = error_plot("LP fractional → rounding infeasible")

        elif lp_state == "lp_infeasible":
            p1 = error_plot("LP infeasible")

        else:
            p1 = error_plot("Unknown LP state")

    except MIPSolverException:
        p1 = error_plot("LP infeasible (solver exception)")
        lp_status = "infeasible"

    H = find_square_root(G)

    if H is None:
        p2 = error_plot("No feasible H")
        has_root = False
        edges_H = None
    else:
        p2 = H.plot(title="Feasible H")
        has_root = True
        edges_H = H.size()

    p3 = G.plot(title="Original G")
    combined = graphics_array([p1, p2, p3])

    summary = {
        "index": idx,
        "vertices_G": n,
        "edges_G": G.size(),
        "edges_H": edges_H,
        "lp_status": lp_status,
        "has_root": has_root,
        "rounding_steps": len(rounding_steps),
    }
    summary["class"] = classify_graph(summary)

    is_comparison_case = (
        (has_root and lp_status == "fractional_infeasible") or
        (not has_root and lp_status == "fractional_rounded")
    )

    comparison = None
    if is_comparison_case:
        comparison = {
            "lp_plot": p1,
            "true_H": H.plot(title="True square root H") if H else error_plot("No H"),
            "G": G.plot(title="Original G"),
            "rounding_steps": rounding_steps,
            "index": idx,
            "class": summary["class"],
        }

    return combined, summary, comparison, is_comparison_case, rounding_steps

def run_task(idx, G):
    return process_graph(idx, G)

tasks = [(i, Graph(G0)) for i, G0 in enumerate(graph_gens.nauty_geng(str(n)))]

results = []
mismatch_results = []

for idx, G in tasks:
    combined, summary, comparison, is_comparison_case, rounding_steps = run_task(idx, G)
    results.append((combined, summary))

    if is_comparison_case and comparison is not None:
        mismatch_results.append((comparison, summary, rounding_steps, G))
    print(
        f"Graph {idx:>4} | "
        f"LP={summary['lp_status']:<10} | "
        f"H={summary['has_root']} | "
        f"Steps={summary['rounding_steps']} | "
        f"{summary['class']}"
    )

with open(summary_file, "w") as f:
    f.write(
        f"{'Graph':>5} | {'Verts':>5} | {'Edges(G)':>8} | "
        f"{'Edges(H)':>8} | {'LP':>10} | {'Root':>5} | "
        f"{'Steps':>5} | Class\n"
    )
    f.write("-" * 100 + "\n")

    for _, s in results:
        f.write(
            f"{s['index']:>5} | "
            f"{s['vertices_G']:>5} | "
            f"{s['edges_G']:>8} | "
            f"{str(s['edges_H']):>8} | "
            f"{s['lp_status']:<10} | "
            f"{'Yes' if s['has_root'] else 'No':>5} | "
            f"{s['rounding_steps']:>5} | "
            f"{s['class']}\n"
        )

print(f"📝 Summary written to {summary_file}")

with PdfPages(output_pdf) as pdf:
    for p, _ in results:
        fig = p.matplotlib(figsize=(10, 10))
        pdf.savefig(fig)
        plt.close(fig)

print(f"✅ Combined {len(results)} graphs into {output_pdf}")

if mismatch_results:
    with PdfPages(comparison_pdf) as pdf:
        for comp, s, steps, G in mismatch_results:
            top = graphics_array(
                [comp["lp_plot"], comp["true_H"], comp["G"]],
                ncols=3
            )
            fig = top.matplotlib(figsize=(12, 4))
            plt.suptitle(f"Graph {comp['index']} — {comp['class']}")
            pdf.savefig(fig)
            plt.close(fig)

            panels = []
            for k, step in enumerate(comp["rounding_steps"], start=1):
                panels.append(
                    plot_with_labels(
                        step["graph"],
                        f"Rounding {k} (fix {step['fixed']}={step['value']})"
                    )
                )

                if len(panels) == 3:
                    page = graphics_array(panels, ncols=3)
                    fig = page.matplotlib(figsize=(12, 4))
                    pdf.savefig(fig)
                    plt.close(fig)
                    panels = []

            if panels:
                page = graphics_array(panels, ncols=3)
                fig = page.matplotlib(figsize=(12, 4))
                pdf.savefig(fig)
                plt.close(fig)

    print(f"⚠️ Comparison + rounding steps saved to {comparison_pdf}")

Graph    0 | LP=integral   | H=True | Steps=1 | OK: H exists, LP integral
Graph    1 | LP=integral   | H=True | Steps=1 | OK: H exists, LP integral
Graph    2 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    3 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    4 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    5 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    6 | LP=integral   | H=True | Steps=1 | OK: H exists, LP integral
Graph    7 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    8 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph    9 | LP=integral   | H=True | Steps=1 | OK: H exists, LP integral
Graph   10 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph   11 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph   12 | LP=infeasible | H=False | Steps=0 | OK: no H, LP infeasible
Graph   13 | LP=infeasible | H=False | Steps=0 

<Figure size 640x480 with 0 Axes>

In [47]:
# Analyze graph summary with rounding

filename = "max_rounding_triangular_graph_output/n=7/graph_summary_n=7.txt"
tree = {
    "total": 0,
    "H_exists": {
        "total": 0,
        "integral": 0,
        "fractional_rounded": 0,
        "fractional_infeasible": 0,
        "lp_infeasible": 0,
    },
    "H_not_exists": {
        "total": 0,
        "integral": 0,
        "fractional_rounded": 0,
        "fractional_infeasible": 0,
        "lp_infeasible": 0,
    },
}

violations = []
rounding_stats = {
    "total": 0,
    "zero_steps": 0,
    "nonzero_steps": 0,
    "max_steps": 0
}

count_feasible_root_no = 0

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith('-') or line.startswith('Graph'):
            continue

        parts = [p.strip() for p in line.split('|')]
        if len(parts) <= 7:
            continue

        graph, verts, edges_g, edges_h, lp, root, steps, *rest = parts
        steps = int(steps)

        H_exists = edges_h.lower() != "none"

        tree["total"] += 1
        rounding_stats["total"] += 1

        if steps == 0:
            rounding_stats["zero_steps"] += 1
        else:
            rounding_stats["nonzero_steps"] += 1
            rounding_stats["max_steps"] = max(
                rounding_stats["max_steps"], steps
            )

        if H_exists:
            tree["H_exists"]["total"] += 1

            if lp == "integral":
                tree["H_exists"]["integral"] += 1

            elif lp == "fractional_rounded":
                tree["H_exists"]["fractional_rounded"] += 1

            elif lp == "fractional_infeasible":
                tree["H_exists"]["fractional_infeasible"] += 1

            elif lp == "infeasible":
                tree["H_exists"]["lp_infeasible"] += 1
                violations.append(parts)

        else:
            tree["H_not_exists"]["total"] += 1

            if lp == "infeasible":
                tree["H_not_exists"]["lp_infeasible"] += 1

            elif lp == "fractional_infeasible":
                tree["H_not_exists"]["fractional_infeasible"] += 1

            elif lp == "fractional_rounded":
                tree["H_not_exists"]["fractional_rounded"] += 1
                violations.append(parts)

            elif lp == "integral":
                tree["H_not_exists"]["integral"] += 1
                violations.append(parts)


print(f"\nFeasible LP & Root=No count: {count_feasible_root_no}")

if violations:
    print("\n⚠️ Violations found (LP infeasible but H exists):")
    for v in violations:
        print(v)
else:
    print("\n✅ All LP infeasible cases imply H does not exist.")

print(f"Total graphs: {tree['total']}\n")

print("H exists:")
for k in ["integral", "fractional_rounded", "fractional_infeasible", "lp_infeasible"]:
    print(f"  {k:22}: {tree['H_exists'][k]}")

print("\nH does NOT exist:")
for k in ["lp_infeasible", "fractional_infeasible", "fractional_rounded", "integral"]:
    print(f"  {k:22}: {tree['H_not_exists'][k]}")


Feasible LP & Root=No count: 0

✅ All LP infeasible cases imply H does not exist.
Total graphs: 1044

H exists:
  integral              : 51
  fractional_rounded    : 24
  fractional_infeasible : 2
  lp_infeasible         : 0

H does NOT exist:
  lp_infeasible         : 831
  fractional_infeasible : 136
  fractional_rounded    : 0
  integral              : 0
